In [1]:
import torch, torch.nn as nn
import torch.nn.functional as F
print(torch.__version__)


2.11.0+cu130


In [2]:
#Cell H1: capacity, in rows per parameter
for name, n in (("A", 650), ("B", 4810), ("C", 17226)):
  print(name, n, round(1077/n, 3))

A 650 1.657
B 4810 0.224
C 17226 0.063


- Which model has the most room to memorise, and what did it do on Day 16 (train accuracy, train loss)?
  * Model C, because it has the most room to memorise: 17226 parameters for 1077 training rows (the fewest rows per parameter). On Day 16 it reached train accuracy 1.0 and train loss 0.0006, while its validation loss stayed at 0.0834.

- Name two ways overfitting in a network differs from overfitting in a random forest (hint: time, and the number of knobs).
  * (1) Time: a network overfits over the epochs (train loss keeps falling while validation loss turns up), so when I stop training matters; a forest is built once and does not get worse as it runs. (2) Knobs: a forest has one main knob (max_depth), while a network has many (depth, width, epochs, learning rate, dropout, weight decay), so there is more to tune and more ways to overfit.


In [3]:
#Cell H2: weight decay with numbers (SGD, Adam, AdamW)
def step_once(opt_class, **kw):
  w = torch.nn.Parameter(torch.tensor([2.0]))
  opt = opt_class([w], lr=0.01, **kw)
  w.grad = torch.tensor([0.5])
  opt.step()
  return round(w.item(), 3)

print("SGD                    ", step_once(torch.optim.SGD))
print("SGD weight_decay=0.5   ", step_once(torch.optim.SGD, weight_decay=0.5))
print("Adam                   ", step_once(torch.optim.Adam))
print("Adam weight_decay=0.5  ", step_once(torch.optim.Adam, weight_decay=0.5))
print("AdamW weight_decay=0.5 ", step_once(torch.optim.AdamW, weight_decay=0.5))

SGD                     1.995
SGD weight_decay=0.5    1.985
Adam                    1.99
Adam weight_decay=0.5   1.99
AdamW weight_decay=0.5  1.98


- "Adam with weight_decay=0.5" gave the same answer as "Adam" without it. Why?
  * Adam with weight_decay adds the penalty (wd x w) to the gradient, and then Adam normalises the step size, so the step is about lr whatever the gradient size (gradients 0.5 and 1.5 both gave 1.9). The extra penalty is divided away, so the result is the same as plain Adam.

- Why is AdamW the right choice when you want weight decay with Adam?
  * AdamW shrinks the weights separately from the gradient (w x (1 - lr x wd)) and then does the Adam step, so the shrink is not normalised away and weight decay really works.


In [4]:
#Cell H3: early stopping by hand
def early_stop(losses, patience):
  best, best_epoch, bad = float("inf"), None, 0
  for epoch, loss in enumerate(losses, 1):
    if loss < best:
      best, best_epoch, bad = loss, epoch, 0
    else:
      bad += 1
      if bad >= patience:
        return epoch, best_epoch, best
  return None, best_epoch, best

In [5]:
losses = [1.0, 0.8, 0.6, 0.55, 0.56, 0.57, 0.58, 0.50, 0.51]
print(early_stop(losses, 3))
print(early_stop(losses, 4))

(7, 4, 0.55)
(None, 8, 0.5)


- With patience 3 the loop stopped at epoch 7 and missed 0.50 at epoch 8. Why does patience 4 find it?
  * because with patience 3: The conditional statement if bad >= patience (3 >= 3) becomes true. The function executes return epoch, best_epoch, best, returning (7, 4, 0.55). The loop terminates instantly, so the code never reads or processes epoch 8. With patience 4: The conditional 3 >= 4 evaluates to false. The loop is allowed to proceed to the next iteration.

- On Day 16, B's validation loss over epochs 25-30 was jagged (0.1112, 0.0973, 0.0970, 0.0897, 0.0873,
0.1061). What does that tell you about choosing the patience?
  * A jagged curve means one bad stretch does not mean the model is finished: a later epoch can still beat the best (like 0.50 at epoch 8 after three bad epochs). A short patience would stop on a random bump, so I should choose a patience longer than the typical wobble (several epochs on B), at the cost of some extra epochs, and restore the best weights.

In [12]:
#Cell H4: two learning-rate schedulers on a toy optimiser
p = torch.nn.Parameter(torch.zeros(1))
opt = torch.optim.SGD([p], lr=0.1)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=3, gamma=0.5)
for epoch in range(1, 8):
  print(epoch, opt.param_groups[0]["lr"])
  opt.step()
  sched.step()
print("\n")

p = torch.nn.Parameter(torch.zeros(1))
opt = torch.optim.SGD([p], lr=0.1)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=1)
for epoch, loss in enumerate([1.0, 0.9, 0.9, 0.9, 0.9, 0.8, 0.8, 0.8], 1):
  print(epoch, loss, opt.param_groups[0]["lr"])
  opt.step()
  sched.step(loss)

1 0.1
2 0.1
3 0.1
4 0.05
5 0.05
6 0.05
7 0.025


1 1.0 0.1
2 0.9 0.1
3 0.9 0.1
4 0.9 0.1
5 0.9 0.05
6 0.8 0.05
7 0.8 0.05
8 0.8 0.05


- Why does StepLR need no metric but ReduceLROnPlateau does?
  * StepLR changes the learning rate on a fixed calendar (multiply by gamma every step_size epochs), so it only counts epochs. ReduceLROnPlateau reacts to how training is going, so it needs the validation loss in sched.step(loss) to know whether there is still improvement.

- In the second loop, why did the learning rate drop only at epoch 5, not at epoch 3?
(Count the epochs without a new best loss, then compare with patience=1.)
  * The best loss (0.9) came at epoch 2. Epoch 3 was the first epoch without a new best (count 1), which is not more than patience=1. Epoch 4 was the second (count 2 > 1), so the scheduler halved the learning rate after epoch 4 and the new 0.05 first shows at epoch 5.

In [13]:
#Cell H5: dropout inside a network
torch.manual_seed(42)
net = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64,10))
print(list(net.state_dict().keys()), sum(p.numel() for p in net.parameters()))

x = torch.rand(3, 64)
net.train()
print(torch.equal(net(x), net(x)))
net.eval()
print(torch.equal(net(x), net(x)))

['0.weight', '0.bias', '3.weight', '3.bias', '6.weight', '6.bias'] 17226
False
True


- Why are the keys numbered 0, 3, 6, and why is the parameter count still 17226 (the same as C)?
  * The keys are the positions of the layers inside nn.Sequential: Linear at 0, ReLU at 1, Dropout at 2, Linear at 3, ReLU at 4, Dropout at 5, Linear at 6. ReLU and Dropout have no weights, so they do not appear in the state_dict, and the count stays 17226 because Dropout adds no parameters; only the three Linear layers count.

- Why do two forward passes on the SAME input differ in train mode but not in eval mode?
  * In train mode Dropout randomly zeroes some units on every pass (and scales the rest), so the same input gives different outputs. In eval mode Dropout is switched off, so the same input always gives the same output.

In [14]:
#Cell H6: augmentation by hand (a tiny 4x4 "image" with ink touching the right edge)
img = torch.zeros(4, 4)
img[1:3, 2:4] = 1.0
print(img.sum().item(), img[:, 3].sum().item())

rolled = torch.roll(img, shifts=(0, 1), dims=(0, 1))
print(rolled.sum().item(), rolled[:, 0].sum().item())

pad = F.pad(img.reshape(1, 4, 4), (1, 1, 1, 1))[0]
shifted = pad[1:5, 0:4]
print(shifted.sum().item(), shifted[:, 0].sum().item())

noisy = (img + 0.1 * torch.randn(4, 4)).clamp(0, 1)
print(noisy.min().item() >= 0, noisy.max().item() <= 1)

4.0 2.0
4.0 2.0
2.0 0.0
True True


- The roll moved the ink from the right edge to the LEFT edge, and the zero-fill shift dropped it.
  * The roll wraps the ink around, so a stroke that left the right edge reappears on the left edge. The zero-fill shift lets the ink fall off the edge and fills the empty column with zeros (blank paper).

- Which one keeps the picture honest, and which one invents a new picture?
  * The zero-fill shift keeps the picture honest: it looks like the same digit written a little off-centre, so the label is still true. The roll invents a new picture: strokes appear at an edge where the digit was never written, so the image may no longer match its label (that is why it was the worst result).

- Why is it safe to augment only the training set, and what would go wrong if you also augmented the
test set?
  * Augmentation is only extra practice for learning. Validation and test must measure the model on real, unseen data. If I also augmented the test set I would be scoring it on altered images, so the number would no longer show how it performs on real digits and could not be compared fairly between runs.


**Cell H7: a map, in your own words (markdown, no code)**

| Technique | What it changes |
|---|---|
| Dropout | model (random units are switched off during training) |
| Weight decay (AdamW) | weights (shrinks them a little every step) |
| Early stopping | training time (stop when validation stops improving) |
| LR scheduling | learning rate |
| Augmentation | data (label-preserving variations of the training images) |

- Which technique do I expect to help C most, and why?
  * I expected augmentation (or dropout) to help most, because C memorises the training rows (train accuracy 1.0, train loss 0.0006) and both stop it from relying on the exact training images.

- What did the lab show?
  * Noise augmentation (std 0.1) helped most: best validation loss 0.0388 against the baseline's 0.0731 (47% lower). Dropout 0.3 came next (0.0450, 38% lower), while the roll augmentation, AdamW 1.0 and dropout 0.5 made it worse.
